# Paper III – Additional Experiments

**Objective**  
This notebook implements the supplementary experiments used to analyse model stability and sensitivity in Flowmetrics. It contains two ablation studies:

- **Ablation Study I – Stage order sensitivity:**  
  Evaluates how predictions change when the Flowmetrics stages are permuted in the prompt.
- **Ablation Study II – Context reduction and component isolation:**  
  Tests how model behaviour shifts when evidence is systematically reduced, probing reliance on specific input components.

**Project:** Flowmetrics – Article III (Predicting Research Impact)  
**Last updated:** 2025-12-05

---

## Task overview

In the paper, research impact modelling is formalised as a two-part task:

1. **Impact stage prediction** – a multi-label classification task where the model selects one or more Flowmetrics stages for a topic pair.  
2. **Structured data-to-text justification** – the model produces a short explanation grounded in the platform evidence.  
   *(Only the first component – stage prediction – is evaluated in this ablation notebook.)*

The goal of the classification task is to determine which Flowmetrics stages are supported by platform-level attention signals for a given pair of research topics. The five stages are:

**Reach, Engagement, Feedback, Influence, Outcome**  
(Feedback is implemented for completeness but unsupported by available evidence.)

### Flowmetrics stages

- **Reach** – Initial visibility of research.  
- **Engagement** – Interactive public or community responses.  
- **Feedback** – Scholarly critique or expert commentary *(unsupported in this dataset)*.  
- **Influence** – Uptake in authoritative or scholarly discourse.  
- **Outcome** – Tangible policy, practice, or technological effects.

These definitions mirror those used in the paper’s Task Definition section.

### Evidence and evaluation

Each model receives:

- the structured topic-pair representation used in the main experiments  
- aggregated and normalised platform co-mention signals from Altmetric and CrossRef  
- a standardised zero-shot prompt describing all five stages

Evaluation follows the three axes used in the paper:

1. **Stage-Level Performance**  
   Precision, recall, F1, and accuracy for identifying supported stages.

2. **Sequence-Level Coherence**  
   Logical ordering relative to the canonical Flowmetrics trajectory  
   *(Reach → Engagement → Influence → Outcome)*.  
   Feedback is omitted from sequence metrics because no evidence supports it.

3. **Cross-Model Agreement**  
   Pairwise agreement in stage selection (Jaccard similarity) and sequence ordering (Kendall’s Tau).

### Models evaluated

`gpt-3.5-turbo`, `gpt-4o`, `gpt-4o-mini`, `claude-3-7-sonnet-20250219`, `claude-3-5-haiku-20241022`, `deepseek-chat`, `open-mixtral-8x22b`

---

## Table of Contents
1. [Objective](#Objective)
2. [Task Overview](#Task-Overview)
3. [Environment Setup](#Environment-Setup)
4. [Data Loading and Filtering](#Data-Loading-and-Filtering)
5. [Common Utilities](#Common-Utilities)
6. [Prompt Assembly](#Prompt-Assembly)
7. [Model Loading](#Model-Loading)
8. [Model Inference](#Model-Inference)
9. [Metrics](#Metrics)
10. [Ablation Studies I – Stage Order Shuffling](#Ablation-Studies-I-–-Stage-Order-Shuffling)
11. [Ablation Studies II – Context and Pipeline Isolation](#Ablation-Studies-II-–-Context-and-Pipeline-Isolation)
12. [Results Aggregation](#Results-Aggregation)
13. [Visualisations](#Visualisations)
14. [Summary of Ablation Results](#Summary)
15. [Discussion](#Discussion)

## Original Prompt (Canonical version)

All ablation experiments use a single **canonical prompt** as the baseline. This prompt defines the task, the stage descriptions, and the expected output format. In Ablation Study I, stage-order permutations are created by modifying only the list of stage definitions while keeping the rest of the prompt constant, allowing us to isolate the effect of stage ordering. In Ablation Study II, the same prompt is systematically reduced or altered to test how model behaviour changes when specific contextual elements are removed. This canonical prompt therefore serves as the control configuration for all ablation conditions.

Below is the exact prompt used in all baseline evaluations:

```text
You are an expert in research impact analysis. Your task is to assess the impact of a pair of research topics based on structured evidence of platform co-mentions and shared concepts.

This is a multi-label classification problem. Your goal is to classify all impact stages as either supported or not, based on the strength and relevance of the evidence for each stage.

---
Impact Stages:

- Reach: Broad dissemination of research to general audiences via mass communication platforms (e.g., Twitter, Facebook, Wikipedia).
- Engagement: Active interaction, discussion, or interpretation of research in community-driven forums (e.g., Blogs, Reddit, YouTube, Mendeley).
- Feedback: Scholarly reactions or critical appraisals, often indicating academic interest (e.g., Peer Review).
- Influence: Contribution to discourse in authoritative contexts (e.g., citations via CrossRef, media coverage via News).
- Outcome: Tangible societal or technological effects arising from research (e.g., Policy documents, Patents).

---
Important Notes:

- Platform values are normalised per platform and impact dimension at the topic level (0.0 = no evidence, 1.0 = maximum evidence). For topic pairs, scores are summed (range: 0.0–2.0) to reflect combined impact.
- Classify all impact stages with meaningful or emerging support, considering the total cumulative evidence across platforms, even if individual platform scores are low.
- Err slightly on the side of inclusiveness: if multiple signals exist across platforms, prefer assigning the stage rather than omitting it.
- If cumulative evidence across all platforms for a stage is strictly zero, assign only the "Reach" stage.

---
Now classify this pair:

- Topic 1: {t_A}
- Topic 2: {t_B}
- Shared Concepts: {shared_concepts}

Platform Co-mention Evidence (normalised values):
{full_platform_lines}

---
You must begin your output exactly with:

Impact Stages with Sufficient Support: [list of stages]

Followed by:

Impact Summary:
Write a concise (2–4 sentence) summary explaining the evidence for each assigned impact stage. Mention the key platforms and shared concepts that support each stage. Highlight how cumulative evidence across platforms contributed to classification, even if individual signals are small.

## Environment Setup

In [30]:
# Environment Setup
import os, sys, json, random
from pathlib import Path

import numpy as np
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS

ablation_dir = Path(".").resolve()
experiments_dir = ablation_dir.parent
repo_root = experiments_dir.parent
sys.path.append(str(experiments_dir))

from config import models, impact_stages, stage_order

random.seed(42)
np.random.seed(42)

data_dir = repo_root / "data"
outputs_dir = ablation_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

GRAPH_FILE = data_dir / "impact_augmented_kg.ttl"
if not GRAPH_FILE.exists():
    raise FileNotFoundError(f"Missing RDF file: {GRAPH_FILE}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

run_cfg = {
    "seed": 42,
    "models": list(models),
    "impact_stages": list(impact_stages),
    "stage_order": list(stage_order),
    "paths": {
        "repo_root": str(repo_root),
        "experiments_dir": str(experiments_dir),
        "ablation_dir": str(ablation_dir),
        "data_dir": str(data_dir),
        "outputs_dir": str(outputs_dir),
        "graph_file": str(GRAPH_FILE),
    },
}
(outputs_dir / "run_config.json").write_text(json.dumps(run_cfg, indent=2), encoding="utf-8")

print("Environment ready")
#print(f"Repo root: {repo_root}")
#print(f"Data dir: {data_dir}")
#print(f"Outputs dir: {outputs_dir}")
print(f"RDF file: {GRAPH_FILE.name}")
print(f"Models: {models}")
print(f"Stages: {impact_stages}")
print(f"Canonical order: {stage_order}")

Environment ready
RDF file: impact_augmented_kg.ttl
Models: ['gpt-3.5-turbo', 'gpt-4o', 'gpt-4o-mini', 'claude-3-7-sonnet-20250219', 'claude-3-5-haiku-20241022', 'open-mixtral-8x22b', 'deepseek-chat']
Stages: ['reach', 'engagement', 'feedback', 'influence', 'outcome']
Canonical order: {'reach': 0, 'engagement': 1, 'feedback': 2, 'influence': 3, 'outcome': 4}


## Data Loading, Filtering, and Stratified Sampling

This notebook operates on the same impact-augmented knowledge graph used in the main experiments. The KG contains 771 topic pairs enriched with platform-level attention signals and Flowmetrics stage annotations derived from cumulative evidence.

To keep the ablation studies computationally feasible while preserving representativeness, we select a **stratified sample of 200 topic pairs**. Sampling is stratified by **stage diversity**—the number of Flowmetrics stages with non-zero evidence for each pair. This ensures that the subset includes:

- simpler cases with two or three supported stages, and  
- more complex cases exhibiting four-stage trajectories.

This stratified sample maintains the range of impact profiles present in the full KG while making it possible to evaluate all seven models across ten prompt permutations in Ablation Study I and multiple context-reduction conditions in Ablation Study II.

In [31]:
# Data Loading, Filtering, and Stratified Sampling
from rdflib import Namespace
from typing import Iterable
from collections import Counter

g = Graph()
g.parse(GRAPH_FILE, format="turtle")
print(f"Graph loaded with {len(g):,} triples")

FLOW = Namespace("http://example.org/flowmetrics#")

def compute_true_stages_column(
    df_pairs: pd.DataFrame,
    stages: Iterable[str],
    threshold: float = 0.003,
    colname: str = "true_stages",
) -> pd.DataFrame:
    stages = list(stages)
    def _row_true(row):
        out = set()
        for st in stages:
            ev = row.get(st, [])
            if isinstance(ev, list):
                total = 0.0
                for it in ev:
                    if isinstance(it, (list, tuple)) and len(it) == 2:
                        try:
                            total += float(it[1])
                        except Exception:
                            pass
                if total > threshold:
                    out.add(st)
        return out
    df_pairs[colname] = df_pairs.apply(_row_true, axis=1)
    return df_pairs

def get_label(node):
    label = g.value(node, RDFS.label)
    return str(label) if label else str(node)

def extract_scores(node):
    return [
        (str(platform), float(g.value(node, FLOW.score)))
        for platform in g.objects(node, FLOW.platform)
        if g.value(node, FLOW.score) is not None
    ]

records = []
for pair in g.subjects(RDF.type, FLOW.TopicPair):
    topics = list(g.objects(pair, FLOW.hasTopic))
    if len(topics) != 2:
        continue
    t1, t2 = topics
    t1_label, t2_label = get_label(t1), get_label(t2)
    shared_concepts = [str(c) for c in g.objects(pair, FLOW.hasSharedConcept)]

    def get_impact(prop):
        return [x for node in g.objects(pair, prop) for x in extract_scores(node)]

    records.append({
        "topic_1_id": str(t1),
        "topic_1_name": t1_label,
        "topic_2_id": str(t2),
        "topic_2_name": t2_label,
        "shared_concepts": shared_concepts,
        "reach": get_impact(FLOW.hasReachImpact),
        "engagement": get_impact(FLOW.hasEngagementImpact),
        "feedback": get_impact(FLOW.hasFeedbackImpact),
        "influence": get_impact(FLOW.hasInfluenceImpact),
        "outcome": get_impact(FLOW.hasOutcomeImpact),
    })

df_full = pd.DataFrame(records)
print(f"Extracted topic pairs: {len(df_full):,}")

df_full = compute_true_stages_column(df_full, impact_stages, threshold=0.003)
df_full["n_stages_true"] = df_full["true_stages"].apply(len)

N_TARGET = 200
strata_counts = df_full["n_stages_true"].value_counts().sort_index()
strata = strata_counts.index.tolist()

if N_TARGET <= 0:
    raise ValueError("N_TARGET must be a positive integer.")

if N_TARGET < len(strata):
    top_strata = strata_counts.sort_values(ascending=False).index[:N_TARGET]
    alloc = {s: (1 if s in top_strata else 0) for s in strata}
else:
    props = strata_counts / strata_counts.sum()
    exact = props * N_TARGET
    alloc = dict(zip(strata, np.floor(exact).astype(int)))
    residual = N_TARGET - sum(alloc.values())
    if residual > 0:
        remainders = (exact - np.floor(exact)).sort_values(ascending=False)
        for s in remainders.index[:residual]:
            alloc[s] += 1

for s in list(alloc.keys()):
    alloc[s] = int(min(alloc[s], (df_full["n_stages_true"] == s).sum()))

while sum(alloc.values()) < N_TARGET:
    for s in strata_counts.sort_values(ascending=False).index:
        avail = (df_full["n_stages_true"] == s).sum()
        if alloc[s] < avail:
            alloc[s] += 1
            if sum(alloc.values()) == N_TARGET:
                break

# Sample
parts = []
rng = 42
for s, n in alloc.items():
    if n <= 0: 
        continue
    pool = df_full[df_full["n_stages_true"] == s]
    parts.append(pool.sample(n=n, random_state=rng))
df = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Publication stratified subset ready.")
print(f"Total sampled: {len(df)} topic pairs (target {N_TARGET})")
print("Allocation by stage diversity:", {int(k): int(v) for k, v in alloc.items()})

print("============ Stratified Subset Summary ============")
print(f"Total topic pairs: {len(df)}")
print("Stage diversity distribution:")
display(df["n_stages_true"].value_counts().sort_index().to_frame("count"))

print(f"\nAverage number of supported stages per pair: {df['n_stages_true'].mean():.2f}")

stage_counts = Counter([st for s in df["true_stages"] for st in s])
stage_df = pd.DataFrame.from_dict(stage_counts, orient="index", columns=["count"]).reindex(impact_stages)
stage_df["proportion"] = stage_df["count"] / stage_df["count"].sum()
print("\nStage-level coverage:")
display(stage_df)

print("\nExample topic pairs per stage-diversity level:")
for d in sorted(df["n_stages_true"].unique()):
    examples = df[df["n_stages_true"] == d].head(2)
    for _, row in examples.iterrows():
        print(f"  • {row['topic_1_name']} ↔ {row['topic_2_name']} ({len(row['true_stages'])} stages: {', '.join(row['true_stages'])})")

Graph loaded with 33,897 triples
Extracted topic pairs: 771
Publication stratified subset ready.
Total sampled: 200 topic pairs (target 200)
Allocation by stage diversity: {0: 14, 1: 20, 2: 21, 3: 34, 4: 111}
============ Stratified Subset Summary ============
Total topic pairs: 200
Stage diversity distribution:


,count
n_stages_true,
0,14
1,20
2,21
3,34
4,111



Average number of supported stages per pair: 3.04

Stage-level coverage:


,count,proportion
reach,181.0,0.297697
engagement,122.0,0.200658
feedback,NaN,NaN
influence,155.0,0.254934
outcome,150.0,0.246711



Example topic pairs per stage-diversity level:
  • audio ↔ automation (0 stages: )
  • prediction modes ↔ facial expression (0 stages: )
  • viewpoint ↔ software frameworks (1 stages: reach)
  • virtual spaces ↔ audio (1 stages: reach)
  • search engines ↔ knowledge base (2 stages: influence, outcome)
  • automation ↔ vehicles (2 stages: reach, outcome)
  • reference image ↔ facial images (3 stages: reach, influence, outcome)
  • encoder-decoder ↔ computational efficiency (3 stages: reach, influence, outcome)
  • user information ↔ integrated data (4 stages: outcome, reach, influence, engagement)
  • adaptive algorithms ↔ reference image (4 stages: outcome, reach, influence, engagement)


## Common Utilities
Helper functions for prompt generation, deterministic shuffling, configuration management, and metrics scaffolding.

In [32]:
# Common Utilities
from collections import defaultdict
STAGES_CANON = list(impact_stages)
STAGE_LC2CAN = {s.lower(): s for s in impact_stages}

def build_shared_concepts_str(concepts, max_terms=None) -> str:
    if not concepts:
        return "(none)"
    items = [str(x) for x in concepts]
    return ", ".join(items[:max_terms]) + (" …" if max_terms and len(items) > max_terms else "")

def assemble_platform_lines_for_pair_flat(row, digits: int = 3) -> str:
    """Flatten evidence across all stages into a single comma-separated list."""
    totals = defaultdict(float)
    for stage in impact_stages:
        entries = row.get(stage, [])
        if not isinstance(entries, list):
            continue
        for item in entries:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                plat, val = item
                try:
                    totals[str(plat)] += float(val)
                except Exception:
                    pass
    # Paper convention: pair score range 0.0–2.0 (cap)
    items = sorted(totals.items(), key=lambda kv: kv[1], reverse=True)
    return ", ".join(f"{p}: {min(v, 2.0):.{digits}f}" for p, v in items)

print("Common utilities ready.")

Common utilities ready.


## Prompt Assembly

This section constructs the full classification prompt for each research topic pair using information extracted from the impact-augmented knowledge graph.

Each prompt instantiates the canonical Flowmetrics template by inserting:
- the two topic labels  
- shared concepts between them  
- normalised platform co-mention evidence aggregated at the topic-pair level  

The resulting prompt defines a multi-label classification task in which the model determines which of the five Flowmetrics stages—Reach, Engagement, Feedback, Influence, Outcome—are supported by the cumulative evidence.

The `apply_prompt_template` function serves as the baseline prompt constructor for all experiments in this notebook. Ablation variants introduced later (e.g., permuted stage order or reduced input context) modify targeted components of this canonical prompt while keeping the rest of the structure unchanged.

In [33]:
# Prompt Assembly
import textwrap

BASE_PROMPT_TEMPLATE = textwrap.dedent("""
You are an expert in research impact analysis. Your task is to assess the impact of a pair of research topics based on structured evidence of platform co-mentions and shared concepts.

This is a multi-label classification problem. Your goal is to classify all impact stages as either supported or not, based on the strength and relevance of the evidence for each stage.

---
Impact Stages:

- Reach: Broad dissemination of research to general audiences via mass communication platforms (e.g., Twitter, Facebook, Wikipedia).
- Engagement: Active interaction, discussion, or interpretation of research in community-driven forums (e.g., Blogs, Reddit, YouTube, Mendeley).
- Feedback: Scholarly reactions or critical appraisals, often indicating academic interest (e.g., Peer Review).
- Influence: Contribution to discourse in authoritative contexts (e.g., citations via CrossRef, media coverage via News).
- Outcome: Tangible societal or technological effects arising from research (e.g., Policy documents, Patents).

---
Important Notes:

- Platform values are normalised per platform and impact dimension at the topic level (0.0 = no evidence, 1.0 = maximum evidence). For topic pairs, scores are summed (range: 0.0–2.0) to reflect combined impact.
- Classify all impact stages with meaningful or emerging support, considering the total cumulative evidence across platforms, even if individual platform scores are low.
- Err slightly on the side of inclusiveness: if multiple signals exist across platforms, prefer assigning the stage rather than omitting it.
- If cumulative evidence across all platforms for a stage is strictly zero, assign only the "Reach" stage.

---
Now classify this pair:

- Topic 1: {topic_1}
- Topic 2: {topic_2}
- Shared Concepts: {shared_concepts}

Platform Co-mention Evidence (normalised values):
{full_platform_lines}

---
You must begin your output exactly with:

Impact Stages with Sufficient Support: [list of stages]

Followed by:

Impact Summary:
Write a concise (2–4 sentence) summary explaining the evidence for each assigned impact stage. Mention the key platforms and shared concepts that support each stage. Highlight how cumulative evidence across platforms contributed to classification, even if individual signals are small.
""").strip()

def apply_prompt_template(row: pd.Series, prompt_template: str = BASE_PROMPT_TEMPLATE) -> str:
    return prompt_template.format(
        topic_1=row["topic_1_name"],
        topic_2=row["topic_2_name"],
        shared_concepts=build_shared_concepts_str(row.get("shared_concepts", [])),
        full_platform_lines=assemble_platform_lines_for_pair_flat(row),
    )

## Model Loading

This section initialises all LLMs used in the ablation experiments through a unified, LangChain-compatible interface.

Both API-based and locally hosted models are supported. The function `load_model(model_name, temperature, max_tokens)` returns a model object with a consistent `.invoke()` or `.predict()` interface, ensuring comparable behaviour across providers.

### Supported Models
- **OpenAI:** `gpt-3.5-turbo`, `gpt-4o`, `gpt-4o-mini`  
- **Anthropic:** `claude-3-7-sonnet-20250219`, `claude-3-5-haiku-20241022`  
- **DeepSeek:** `DeepSeek-V3`, `deepseek-reasoner`, `deepseek-chat`  
- **Mistral:** `Mixtral-8x22B`, `open-mixtral-8x7b`  
- **Local models:** e.g., `DeepSeek-R1-Distill-Qwen-1.5B`, `Llama-3.2-1B`, `Mistral-7B-Instruct-v0.1`

API models require their corresponding environment variables (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `DEEPSEEK_API_KEY`, `MISTRAL_API_KEY`). Local models are loaded from the directory defined in `MODEL_PATH`.

This modular loading layer ensures consistent inference settings across all ablation conditions and supports reproducibility of results.

In [35]:
# Model Loading
def load_model(model_name: str, temperature: float = 0.1, max_tokens: int = 1024):
    if model_name in {"gpt-3.5-turbo", "gpt-4o", "gpt-4o-mini"}:
        from langchain.chat_models import ChatOpenAI
        api_key = OPENAI_API_KEY
        if not api_key:
            raise EnvironmentError("OPENAI_API_KEY not set.")
        return ChatOpenAI(model_name=model_name, temperature=temperature, max_tokens=max_tokens, openai_api_key=api_key)

    if model_name in {"claude-3-7-sonnet-20250219", "claude-3-5-haiku-20241022"}:
        from langchain_anthropic import ChatAnthropic
        api_key = ANTHROPIC_API_KEY
        if not api_key:
            raise EnvironmentError("ANTHROPIC_API_KEY not set.")
        return ChatAnthropic(model_name=model_name, temperature=temperature, max_tokens=max_tokens, anthropic_api_key=api_key)

    if model_name in {"DeepSeek-V3", "deepseek-chat", "deepseek-reasoner"}:
        from langchain.chat_models import ChatOpenAI
        api_key = DEEPSEEK_API_KEY
        if not api_key:
            raise EnvironmentError("DEEPSEEK_API_KEY not set.")
        return ChatOpenAI(
            model_name="deepseek-chat",
            temperature=temperature,
            max_tokens=max_tokens,
            openai_api_key=api_key,
            openai_api_base="https://api.deepseek.com/v1"
        )

    if model_name in {"Mixtral 8x22B", "open-mixtral-8x22b", "open-mixtral-8x7b"}:
        from langchain_mistralai import ChatMistralAI
        api_key = MISTRAL_API_KEY
        if not api_key:
            raise EnvironmentError("MISTRAL_API_KEY not set.")
        resolved = "open-mixtral-8x22b" if "22" in model_name else "open-mixtral-8x7b"
        return ChatMistralAI(model_name=resolved, temperature=temperature, max_tokens=max_tokens, mistral_api_key=api_key)

    if model_name.startswith("local:"):
        from transformers import pipeline as hf_pipeline
        from langchain.llms import HuggingFacePipeline
        model_dir = Path(os.getenv("MODEL_PATH", "")) / model_name.split("local:", 1)[-1]
        if not model_dir.exists():
            raise FileNotFoundError(f"Local model not found: {model_dir}")
        gen = hf_pipeline("text-generation", model=str(model_dir), max_new_tokens=max_tokens)
        return HuggingFacePipeline(pipeline=gen)

    raise ValueError(f"Unsupported model: {model_name}")

## Model Inference and Response Parsing

This section runs the constructed prompts through each model and extracts the structured components needed for evaluation.

The inference pipeline includes:
1. **Prompt construction** using `apply_prompt_template()` or the corresponding ablated variant  
2. **Model invocation** via `generate_response(model_name, prompt)`  
3. **Response parsing** to extract:
   - `predicted_stages_[model]`: the list of impact stages identified as supported  
   - `impact_summary_[model]`: the short evidence-grounded explanation  
   - `raw_response_[model]`: the full model output for auditability  

The parser locates the line starting with  
`Impact Stages with Sufficient Support: [...]`  
and reads the narrative following `Impact Summary:`.

These structured outputs provide the basis for analysing stage-level accuracy, sequence coherence, and robustness across the ablation conditions.

In [36]:
# Model Inference & Parsing
import re, time
from httpx import HTTPStatusError
from langchain import LLMChain, PromptTemplate

HEADER_RE = re.compile(
    r"(?:Impact Stages with Sufficient Support|Impact Stages|Supported Stages)\s*:\s*(?:\[(?P<inside1>[^\]]*)\]|(?P<inside2>.+?))(?:\n{2,}|\r?\n|\Z)",
    re.I | re.S,
)
TOKEN_SPLIT = re.compile(r"[,\n;•·\-–—]+")
STAGE_LC2CAN = {s.lower(): s for s in impact_stages}
SUMMARY_RE = re.compile(r"Impact Summary\s*:\s*(.*)$", re.I | re.S)

def robust_parse_supported_stages(text: str) -> list[str]:
    if not text:
        return []
    m = HEADER_RE.search(text)
    if not m:
        return []
    inside = m.group("inside1") or m.group("inside2") or ""
    parts = [t.strip(" .:|/\\()[]{}\"'").lower() for t in TOKEN_SPLIT.split(inside) if t.strip()]
    out, seen = [], set()
    for p in parts:
        s = STAGE_LC2CAN.get(p)
        if s and s not in seen:
            seen.add(s); out.append(s)
    return out

def parse_impact_summary(raw_text: str) -> str:
    m = SUMMARY_RE.search(raw_text or "")
    return m.group(1).strip() if m else ""

def generate_response(model_name: str, prompt: str, *, temperature: float = 0.1, max_tokens: int = 4096) -> str:
    """Call model with retry handling for rate limits (HTTP 429)."""
    model = load_model(model_name, temperature=temperature, max_tokens=max_tokens)
    chain = LLMChain(llm=model, prompt=PromptTemplate(template="{prompt}", input_variables=["prompt"]))
    for attempt in range(3):  # up to 3 retries
        try:
            return chain.run({"prompt": prompt})
        except HTTPStatusError as e:
            if e.response.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"Rate limit hit ({model_name}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
        except Exception as e:
            print(f"Unexpected error ({model_name}): {e}")
            if attempt < 2:
                wait = 5 * (attempt + 1)
                print(f"Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed after 3 attempts for {model_name}")

def iterate_and_generate_responses(
    topic_pairs_df: pd.DataFrame,
    model_name: str,
    *,
    prompt_fn,
    temperature: float = 0.1,
    max_tokens: int = 1024,
) -> pd.DataFrame:
    pred_col = f"predicted_stages_{model_name}"
    summ_col = f"impact_summary_{model_name}"
    raw_col  = f"raw_response_{model_name}"

    raws, preds, sums = [], [], []
    for _, row in topic_pairs_df.iterrows():
        prompt = prompt_fn(row)
        raw = generate_response(model_name, prompt, temperature=temperature, max_tokens=max_tokens).strip()
        raws.append(raw)
        preds.append(robust_parse_supported_stages(raw))
        sums.append(parse_impact_summary(raw))

    out = topic_pairs_df.copy()
    out[pred_col], out[summ_col], out[raw_col] = preds, sums, raws
    #print(f"[{model_name}] empty prediction rate: {out[pred_col].apply(lambda x: not x).mean():.3f}")
    return out

## Metrics - Stage-level performance

This section computes **stage-level performance metrics** for each model under the ablation conditions.

Since no human-labelled gold standard exists, we use the same **proxy reference labels** adopted in the main paper: a stage is treated as *supported* when its cumulative platform evidence exceeds the threshold **0.003**.

Predictions (`predicted_stages_<model>`) are evaluated against these labels using standard **multi-label classification metrics**:

- **Precision** – proportion of predicted stages that are correctly supported  
- **Recall** – proportion of supported stages successfully identified  
- **F1-score** – harmonic mean of precision and recall  
- **Accuracy** – per-stage classification accuracy  

Metrics are computed:
- **Per impact stage** (Reach, Engagement, Feedback, Influence, Outcome), and  
- **Per model–ablation configuration** (baseline prompt, permuted orderings, reduced context variants, etc.)

This cell records all raw metric outputs, which are later aggregated to assess stability and sensitivity across ablation settings.

In [37]:
# Metrics — Stage-Level
from typing import Iterable, Tuple, Dict, List
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, hamming_loss

def _to_one_hot(labels: Iterable[str], stages: List[str]) -> List[int]:
    ls = {x.lower() for x in (labels or [])}
    return [int(st.lower() in ls) for st in stages]

def evaluate_stage_level_one_model(
    df_pairs: pd.DataFrame,
    model_name: str,
    stages: Iterable[str],
    true_col: str = "true_stages",
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    stages = list(stages)
    y_true_sets = df_pairs[true_col].apply(lambda s: {x.lower() for x in (s or set())})

    pred_col = f"predicted_stages_{model_name}"
    if pred_col in df_pairs.columns:
        y_pred_sets = df_pairs[pred_col].apply(
            lambda lst: {x.lower() for x in (lst if isinstance(lst, (list, tuple)) else []) if isinstance(x, str)}
        )
    else:
        y_pred_sets = pd.Series([set() for _ in range(len(df_pairs))], index=df_pairs.index)

    rows = {}
    for st in stages:
        k = st.lower()
        yt = y_true_sets.apply(lambda s: k in s).astype(int)
        yp = y_pred_sets.apply(lambda s: k in s).astype(int)
        rows[st] = {
            "Precision": precision_score(yt, yp, zero_division=0),
            "Recall":    recall_score(yt, yp, zero_division=0),
            "F1":        f1_score(yt, yp, zero_division=0),
            "Accuracy":  accuracy_score(yt, yp),
        }
    per_stage_df = pd.DataFrame(rows).T.loc[stages]

    y_true_bin = np.asarray([_to_one_hot(s or set(), stages) for s in df_pairs[true_col]])
    y_pred_bin = np.asarray([
        _to_one_hot(set(x if isinstance(x, (list, tuple)) else []), stages)
        for x in df_pairs.get(pred_col, [[]] * len(df_pairs))
    ])
    ham = hamming_loss(y_true_bin, y_pred_bin)

    global_dict = {
        "model": model_name,
        "Precision": round(per_stage_df["Precision"].mean(), 3),
        "Recall":    round(per_stage_df["Recall"].mean(), 3),
        "F1 Score":  round(per_stage_df["F1"].mean(), 3),
        "Accuracy":  round(per_stage_df["Accuracy"].mean(), 3),
        "Hamming Loss": round(ham, 3),
        "n_pairs": len(df_pairs),
    }
    return per_stage_df, global_dict

def evaluate_stage_level(
    df_pairs: pd.DataFrame,
    models_list: Iterable[str],
    stages: Iterable[str],
    true_col: str = "true_stages",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    per_stage_all, global_all = [], []
    for m in models_list:
        per_stage_df, global_dict = evaluate_stage_level_one_model(df_pairs, m, stages, true_col=true_col)
        per_stage_df = per_stage_df.reset_index().rename(columns={"index": "stage"})
        per_stage_df.insert(0, "model", m)
        per_stage_all.append(per_stage_df)
        global_all.append(global_dict)
    return (
        pd.concat(per_stage_all, ignore_index=True) if per_stage_all else pd.DataFrame(),
        pd.DataFrame(global_all).sort_values("model") if global_all else pd.DataFrame(),
    )

## Ablation I – Stage order shuffling

**Aim**  
Assess whether LLMs base their predictions on the *semantic meaning* of the Flowmetrics stages or whether they depend on the *order* in which those stages appear in the prompt.

**Method**  
We construct a set of alternative stage-order configurations, including:
- the canonical Flowmetrics ordering,
- several adversarial permutations that place later stages first, and
- multiple random permutations.

Each permutation is injected into the canonical prompt template, with all other components held constant (topic-pair inputs, evidence values, and instructions). All seven models are re-run on the stratified 200-pair subset.

For every model and permutation, we compute:
- `F1(permutation)` – the stage-level F1 score under that permutation  
- `F1(canonical)` – the F1 score under the original ordering  

The **sensitivity measure** is defined as:

`ΔF1 = F1(permutation) − F1(canonical)`

This yields:
- **model-level statistics** (mean, std, min, max ΔF1), and  
- a **permutation × model matrix** of mean ΔF1 values.

**Outcome**  
If a model truly understands the stage definitions, ΔF1 should remain close to zero across permutations. Large positive or negative ΔF1 values indicate that the model relies on the specific ordering of the stage list rather than on the underlying evidence, signalling prompt-order sensitivity.

In [12]:
# Ablation I — Stage Order Shuffling (Stage-Level; canonical + adversarial + random)
from typing import List, Tuple
import numpy as np, pandas as pd, re
from tqdm import tqdm

RESULTS_DIR_I = outputs_dir / "ablation_i_shuffle"
RESULTS_DIR_I.mkdir(parents=True, exist_ok=True)

# Config
SEED_BASE   = 2025
TEMPERATURE = 0.1
MAX_TOKENS  = 4096
NAMES_ONLY  = False
UNORDERED_NOTE = True
RUN_MODELS  = models          # e.g. ["gpt-4o", "gpt-4o-mini"]
N_RANDOM    = 5               # number of additional random permutations
SAMPLE_N    = 200             # number of topic pairs to process (None = all)

# Stage definitions
STAGE_DEFS = {
    "Reach": "- Reach: Broad dissemination of research to general audiences via mass communication platforms (e.g., Twitter, Facebook, Wikipedia).",
    "Engagement": "- Engagement: Active interaction, discussion, or interpretation of research in community-driven forums (e.g., Blogs, Reddit, YouTube).",
    "Feedback": "- Feedback: Scholarly reactions or critical appraisals, often indicating academic interest (e.g., Mendeley).",
    "Influence": "- Influence: Contribution to discourse in authoritative contexts (e.g., citations via CrossRef, media coverage via News).",
    "Outcome": "- Outcome: Tangible societal or technological effects arising from research (e.g., Policy documents, Patents).",
}
_IMPACT_RE = re.compile(r"(Impact Stages:\s*\n)(.*?)(\n---)", flags=re.DOTALL | re.IGNORECASE)

def _stage_block(order: List[str], names_only: bool = False, unordered_note: bool = True) -> str:
    if names_only:
        lines = [f"- {s}" for s in order]
    else:
        lines = [STAGE_DEFS[s.capitalize()] for s in order]
    note = "\n\nNote: Stages are unordered; do not assume this list implies a sequence." if unordered_note else ""
    return "\n".join(lines) + note

def inject_stage_order_into_template(
    base_template: str,
    order: List[str],
    names_only: bool = False,
    unordered_note: bool = True,
) -> str:
    block = f"Impact Stages:\n\n{_stage_block(order, names_only, unordered_note)}\n\n---"
    return _IMPACT_RE.sub(lambda m: block, base_template)

# Permutation generation
ADVERSARIAL = [
    ("perm_rev",        ["Outcome","Influence","Feedback","Engagement","Reach"]),
    ("perm_eng_first",  ["Engagement","Reach","Feedback","Influence","Outcome"]),
    ("perm_inf_first",  ["Influence","Reach","Engagement","Feedback","Outcome"]),
    ("perm_reach_last", ["Engagement","Feedback","Influence","Outcome","Reach"]),
]

def make_permutations(stage_list, k_random: int = 4, seed: int = 2025):
    rng = np.random.RandomState(seed)
    out = [("perm_0_canonical", stage_list[:])] + ADVERSARIAL[:]
    seen = {tuple(p) for _, p in out}
    i = 0
    while i < k_random:
        s = stage_list[:]
        rng.shuffle(s)
        t = tuple(s)
        if t not in seen:
            out.append((f"perm_rand_{i}", s))
            seen.add(t)
            i += 1
    return out

df = compute_true_stages_column(df, STAGES_CANON, threshold=0.003)

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df_base = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
else:
    df_base = df.copy()

print(f"Running Ablation I on {len(df_base)} topic pairs (SAMPLE_N={SAMPLE_N})")

perm_orders = make_permutations(list(stage_order), k_random=N_RANDOM, seed=SEED_BASE)
perm_orders = [(n, [s.capitalize() for s in order]) for n, order in perm_orders]
print(f"Generated {len(perm_orders)} stage permutations:")
for name, order in perm_orders:
    print(f" - {name}: {order}")

all_stage_per_stage, all_stage_global = [], []

for perm_name, order in perm_orders:
    out_dir   = RESULTS_DIR_I / perm_name
    per_path  = out_dir / "stage_level_per_stage.csv"
    glob_path = out_dir / "stage_level_global.csv"

    if per_path.exists() and glob_path.exists():
        print(f"\n=== {perm_name}: cached results found, reusing. ===")
        per_stage = pd.read_csv(per_path)
        global_summary = pd.read_csv(glob_path)

        per_stage["condition"] = f"I:{perm_name}"
        if "presented_order" not in per_stage.columns:
            per_stage["presented_order"] = [tuple(order)] * len(per_stage)
        global_summary["condition"] = f"I:{perm_name}"

        pbar = tqdm(total=len(RUN_MODELS), desc=f"{perm_name} models (cached)")
        pbar.update(len(RUN_MODELS))
        pbar.close()

        all_stage_per_stage.append(per_stage)
        all_stage_global.append(global_summary)
        continue

    print(f"\n=== Running permutation: {perm_name} | order={order} ===")
    templ = inject_stage_order_into_template(
        BASE_PROMPT_TEMPLATE,
        order,
        names_only=NAMES_ONLY,
        unordered_note=UNORDERED_NOTE,
    )

    df_work = df_base.copy()
    models_present = []

    for m in tqdm(RUN_MODELS, desc=f"{perm_name} models"):
        df_work = iterate_and_generate_responses(
            df_work,
            m,
            prompt_fn=lambda row, _templ=templ: apply_prompt_template(row, prompt_template=_templ),
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        if f"predicted_stages_{m}" in df_work.columns:
            models_present.append(m)

    if not models_present:
        print(f"No predictions captured for {perm_name}.")
        continue

    per_stage, global_summary = evaluate_stage_level(
        df_work, models_present, STAGES_CANON, true_col="true_stages"
    )
    per_stage["condition"] = f"I:{perm_name}"
    per_stage["presented_order"] = [tuple(order)] * len(per_stage)
    global_summary["condition"] = f"I:{perm_name}"

    out_dir.mkdir(parents=True, exist_ok=True)
    per_stage.to_csv(per_path, index=False)
    global_summary.to_csv(glob_path, index=False)

    all_stage_per_stage.append(per_stage)
    all_stage_global.append(global_summary)

if all_stage_per_stage:
    abl1_stage_per_stage = pd.concat(all_stage_per_stage, ignore_index=True)
    abl1_stage_global    = pd.concat(all_stage_global, ignore_index=True)

    abl1_stage_per_stage.to_csv(
        RESULTS_DIR_I / "ALL_stage_level_per_stage.csv",
        index=False,
    )
    abl1_stage_global.to_csv(
        RESULTS_DIR_I / "ALL_stage_level_global.csv",
        index=False,
    )

    print("\nSaved Ablation I Stage-Level tables.")
else:
    print("No Ablation I results produced.")

Running Ablation I on 200 topic pairs (SAMPLE_N=200)
Generated 10 stage permutations:
 - perm_0_canonical: ['Reach', 'Engagement', 'Feedback', 'Influence', 'Outcome']
 - perm_rev: ['Outcome', 'Influence', 'Feedback', 'Engagement', 'Reach']
 - perm_eng_first: ['Engagement', 'Reach', 'Feedback', 'Influence', 'Outcome']
 - perm_inf_first: ['Influence', 'Reach', 'Engagement', 'Feedback', 'Outcome']
 - perm_reach_last: ['Engagement', 'Feedback', 'Influence', 'Outcome', 'Reach']
 - perm_rand_0: ['Engagement', 'Influence', 'Reach', 'Outcome', 'Feedback']
 - perm_rand_1: ['Engagement', 'Feedback', 'Reach', 'Outcome', 'Influence']
 - perm_rand_2: ['Influence', 'Outcome', 'Feedback', 'Engagement', 'Reach']
 - perm_rand_3: ['Influence', 'Feedback', 'Outcome', 'Reach', 'Engagement']
 - perm_rand_4: ['Feedback', 'Engagement', 'Influence', 'Reach', 'Outcome']

=== perm_0_canonical: cached results found, reusing. ===


perm_0_canonical models (cached): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 166818.91it/s]



=== perm_rev: cached results found, reusing. ===


perm_rev models (cached): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 175809.15it/s]



=== perm_eng_first: cached results found, reusing. ===


...


=== Running permutation: perm_rand_3 | order=['Influence', 'Feedback', 'Outcome', 'Reach', 'Engagement'] ===


perm_rand_3 models: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [1:13:02<00:00, 626.00s/it]



=== Running permutation: perm_rand_4 | order=['Feedback', 'Engagement', 'Influence', 'Reach', 'Outcome'] ===


perm_rand_4 models: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [1:14:14<00:00, 636.42s/it]


Saved Ablation I Stage-Level tables.


In [23]:
RESULTS_DIR_I = outputs_dir / "ablation_i_shuffle"
path_global = RESULTS_DIR_I / "ALL_stage_level_global.csv"

df_global = pd.read_csv(path_global)
df_global["condition"] = df_global["condition"].astype(str)

canonical = df_global[df_global["condition"] == "I:perm_0_canonical"].copy()
f1_canon = dict(zip(canonical["model"], canonical["F1 Score"]))

df_global["F1 ScoreΔ"] = df_global.apply(
    lambda row: row["F1 Score"] - f1_canon.get(row["model"], np.nan),
    axis=1,
)

abl1_with_delta = df_global.copy()
df_noncanon = abl1_with_delta[abl1_with_delta["condition"] != "I:perm_0_canonical"].copy()

stats_by_model = (
    df_noncanon.groupby("model")["F1 ScoreΔ"]
    .agg(["mean", "std", "min", "max"])
    .sort_index()
    .round(3)
)

table_perm_model = (
    df_noncanon.pivot_table(
        index="condition",
        columns="model",
        values="F1 ScoreΔ",
        aggfunc="mean"
    )
    .sort_index()
    .round(3)
)

print("\nΔF1 stats by model (mean/std/min/max)\n")
print(stats_by_model)

print("\nΔF1 breakdown by permutation × model (mean ΔF1)\n")
print(table_perm_model)


ΔF1 stats by model (mean/std/min/max)

                             mean    std    min    max
claude-3-5-haiku-20241022  -0.201  0.072 -0.251 -0.044
claude-3-7-sonnet-20250219  0.022  0.053 -0.048  0.133
deepseek-chat              -0.064  0.061 -0.153  0.025
gpt-3.5-turbo              -0.028  0.090 -0.178  0.150
gpt-4o                     -0.074  0.064 -0.168  0.035
gpt-4o-mini                 0.062  0.116 -0.080  0.248
open-mixtral-8x22b         -0.016  0.070 -0.114  0.121

ΔF1 breakdown by permutation × model (mean ΔF1)

                   claude-3-5-haiku-20241022  claude-3-7-sonnet-20250219  deepseek-chat  gpt-3.5-turbo  gpt-4o  gpt-4o-mini  open-mixtral-8x22b
I:perm_eng_first                      -0.251                      -0.048         -0.153         -0.178  -0.168       -0.080              -0.114
I:perm_inf_first                      -0.172                      -0.027         -0.068         -0.018  -0.068        0.116              -0.021
I:perm_rand_0                         

## Key Observations

1. **Models cluster clearly into robustness tiers when stage order is perturbed.**  
   - **Most robust:**  
     - **Claude 3.7 Sonnet** (mean +0.022, min −0.048, max +0.133): the only model with a positive mean and tightly bounded variability.  
     - **Mixtral 8x22B** (mean −0.016, min −0.114, max +0.121): small average drop and moderate spread.  
   - **Moderately robust:**  
     - **DeepSeek-Chat** (mean −0.064, min −0.153, max +0.025): mild average degradation with occasional improvements.  
     - **GPT-4o** (mean −0.074, min −0.168, max +0.035): stable overall, though sensitive under specific permutations.  
   - **Least robust or most volatile:**  
     - **Claude 3.5 Haiku** (mean −0.201, min −0.251): systematic degradation, with all ΔF1 values negative.  
     - **GPT-4o-mini** (mean +0.062, min −0.080, max +0.248): highest variance (std 0.116) and extreme swings in both directions.  
     - **GPT-3.5-Turbo** (mean −0.028, min −0.178, max +0.150): wide oscillations, indicating strong sensitivity to ordering.

2. **The most harmful permutations are those that disrupt canonical early-stage placement.**  
   Conditions like `perm_eng_first` and `perm_rand_4` produce the worst drops for several models:  
   - Haiku reaches **−0.251**,  
   - DeepSeek reaches **−0.153**,  
   - GPT-4o reaches **−0.168**.  
   These patterns show that models often depend on predictable early-stage positioning within the prompt.

3. **Certain permutations unexpectedly improve performance for specific models.**  
   - Sonnet improves under `perm_rand_0` (+0.058) and `perm_rev` (+0.133).  
   - GPT-4o-mini shows strong gains under `perm_rev` (+0.248) and `perm_rand_0` (+0.195).  
   - GPT-3.5-Turbo peaks under `perm_rev` (+0.150).  
   These boosts indicate that models can exploit alternative orderings rather than strictly relying on semantic understanding.

4. **Stable models appear to rely on semantic definitions, while unstable models depend on surface order.**  
   Sonnet and Mixtral remain close to zero mean ΔF1, with constrained ranges, suggesting they interpret stages semantically. Conversely, Haiku’s uniformly negative values and GPT-4o-mini’s high variance show dependence on the canonical structure.

**Overall:** Ablation I demonstrates that stage-order changes impact models in markedly different ways. Some models (Sonnet, Mixtral) maintain stable predictions across permutations, while others either degrade consistently (Haiku) or fluctuate unpredictably (GPT-4o-mini, GPT-3.5-Turbo). The evidence shows that ordering effects remain a significant factor in prompt-based Flowmetrics reasoning, and robustness cannot be assumed across model families.

## Ablation II – Context and pipeline isolation

**Aim**  
Assess how different components of the Flowmetrics prompt contribute to model performance. This ablation isolates the effects of structured evidence, stage definitions, and minimal topic-level context to determine which inputs LLMs rely on most.

**Method**  
We design a set of controlled prompt variants that progressively remove or restrict information:

- **Minimal context**  
  Removes stage definitions and all platform evidence, leaving only topic names and shared concepts.
- **Names-only**  
  Includes the five stage names and full evidence, but removes all semantic descriptions of the stages.
- **Selective-evidence prompts**  
  Use the full canonical template but supply evidence for only one stage at a time (Reach, Engagement, Feedback, Influence, or Outcome).
- **Full evidence (baseline)**  
  The original Flowmetrics prompt with full stage definitions and complete platform evidence.

All seven models are re-run on the same stratified sample of 200 topic pairs. For each model and condition, we compute:

- `F1(condition)` – the stage-level F1 under that ablation  
- `F1(full_all)` – the F1 score using the complete prompt  

The **sensitivity measure** is defined as:

`ΔF1 = F1(condition) − F1(full_all)`

This produces:
- **model-level statistics** (mean, std, min, max ΔF1), and  
- a **condition × model matrix** of mean ΔF1 values.

**Outcome**  
This setup clarifies how dependent each model is on specific components of the prompt. Large negative ΔF1 values indicate strong reliance on the removed information (e.g., stage definitions or certain evidence channels). Small or stable ΔF1 values indicate robustness to reduced context or simplified evidence.

In [38]:
# Ablation II — Context and Pipeline Isolation
import re

RESULTS_DIR_II = outputs_dir / "ablation_ii_context"
RESULTS_DIR_II.mkdir(parents=True, exist_ok=True)

TEMPERATURE = 0.1
MAX_TOKENS = 4096
RUN_MODELS = models
SAMPLE_N = 200

def assemble_platform_lines_for_pair_filtered(
    row,
    include_stages: List[str] | None,
    digits: int = 3,
    *,
    flatten: bool = True,
) -> str:
    if flatten:
        keep = {s.lower() for s in include_stages} if include_stages else None
        totals = defaultdict(float)
        for stage in impact_stages:
            if keep and stage.lower() not in keep:
                continue
            entries = row.get(stage, [])
            if not isinstance(entries, list):
                continue
            for it in entries:
                if isinstance(it, (list, tuple)) and len(it) == 2:
                    p, v = it
                    try:
                        totals[str(p)] += float(v)
                    except Exception:
                        pass
        items = sorted(totals.items(), key=lambda kv: kv[1], reverse=True)
        return ", ".join(f"{p}: {min(v, 2.0):.{digits}f}" for p, v in items) if items else "- (no evidence)"
    else:
        keep = {s.lower() for s in include_stages} if include_stages else None
        lines = []
        for stage in impact_stages:
            if keep and stage.lower() not in keep:
                continue
            entries = row.get(stage, [])
            if not isinstance(entries, list):
                continue
            parts = []
            for it in entries:
                if isinstance(it, (list, tuple)) and len(it) == 2:
                    p, v = it
                    try:
                        parts.append(f"{p}: {float(v):.{digits}f}")
                    except Exception:
                        pass
            if parts:
                lines.append(", ".join(parts))
        return "\n".join(lines) if lines else "- (no evidence)"

MINIMAL_PROMPT = """
You are an expert in research impact analysis.
Classify which impact stages apply to the topic pair below.

Now classify this pair:
- Topic 1: {topic_1}
- Topic 2: {topic_2}
- Shared Concepts: {shared_concepts}

Output format:
Impact Stages with Sufficient Support: [list of stages]
Impact Summary: brief justification.
""".strip()

NAMES_ONLY_PROMPT = """
You are an expert in research impact analysis.
Classify which impact stages apply to the topic pair below.

Impact Stages:
- Reach
- Engagement
- Feedback
- Influence
- Outcome

Now classify this pair:
- Topic 1: {topic_1}
- Topic 2: {topic_2}
- Shared Concepts: {shared_concepts}

Platform Co-mention Evidence:
{platform_evidence}

Output format:
Impact Stages with Sufficient Support: [list of stages]
Impact Summary: brief justification.
""".strip()

FULL_PROMPT = BASE_PROMPT_TEMPLATE

def prompt_minimal(row: pd.Series) -> str:
    return MINIMAL_PROMPT.format(
        topic_1=row["topic_1_name"],
        topic_2=row["topic_2_name"],
        shared_concepts=build_shared_concepts_str(row.get("shared_concepts", [])),
    )

def prompt_names_only(row: pd.Series, include_stages: List[str] | None, *, flatten=True) -> str:
    ev = assemble_platform_lines_for_pair_filtered(row, include_stages=include_stages, flatten=flatten)
    return NAMES_ONLY_PROMPT.format(
        topic_1=row["topic_1_name"],
        topic_2=row["topic_2_name"],
        shared_concepts=build_shared_concepts_str(row.get("shared_concepts", [])),
        platform_evidence=ev,
    )

def prompt_full(row: pd.Series, include_stages: List[str] | None, *, flatten=True) -> str:
    ev = assemble_platform_lines_for_pair_filtered(row, include_stages=include_stages, flatten=flatten)
    return FULL_PROMPT.format(
        topic_1=row["topic_1_name"],
        topic_2=row["topic_2_name"],
        shared_concepts=build_shared_concepts_str(row.get("shared_concepts", [])),
        full_platform_lines=ev,
    )

CONDITIONS: Dict[str, Dict] = {
    "II:minimal_no_evidence":   {"builder": lambda row: prompt_minimal(row)},
    "II:names_only_all":        {"builder": lambda row: prompt_names_only(row, include_stages=None,        flatten=True)},
    "II:full_all":              {"builder": lambda row: prompt_full(row,        include_stages=None,        flatten=True)},
    "II:full_only_reach":       {"builder": lambda row: prompt_full(row,        include_stages=["Reach"],      flatten=True)},
    "II:full_only_engagement":  {"builder": lambda row: prompt_full(row,        include_stages=["Engagement"], flatten=True)},
    "II:full_only_feedback":    {"builder": lambda row: prompt_full(row,        include_stages=["Feedback"],   flatten=True)},
    "II:full_only_influence":   {"builder": lambda row: prompt_full(row,        include_stages=["Influence"],  flatten=True)},
    "II:full_only_outcome":     {"builder": lambda row: prompt_full(row,        include_stages=["Outcome"],    flatten=True)},
}

df = compute_true_stages_column(df, STAGES_CANON, threshold=0.003)

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df_base = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
else:
    df_base = df.copy()

print(f"Running Ablation II on {len(df_base)} topic pairs (SAMPLE_N={SAMPLE_N})")

all_per_stage, all_global = [], []

for cond_name, spec in CONDITIONS.items():
    print(f"\n=== {cond_name} ===")
    out_dir = RESULTS_DIR_II / cond_name
    per_path = out_dir / "stage_level_per_stage.csv"
    glob_path = out_dir / "stage_level_global.csv"

    if per_path.exists() and glob_path.exists():
        per_stage = pd.read_csv(per_path)
        global_summary = pd.read_csv(glob_path)

        per_stage["condition"] = cond_name
        global_summary["condition"] = cond_name

        pbar = tqdm(total=len(RUN_MODELS), desc=f"{cond_name} models (cached)")
        pbar.update(len(RUN_MODELS))
        pbar.close()

        all_per_stage.append(per_stage)
        all_global.append(global_summary)
        continue

    df_work = df_base.copy()
    models_present = []

    for m in tqdm(RUN_MODELS, desc=f"{cond_name} models"):
        df_work = iterate_and_generate_responses(
            df_work,
            m,
            prompt_fn=spec["builder"],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        if f"predicted_stages_{m}" in df_work.columns:
            models_present.append(m)

    if not models_present:
        print("No predictions captured for this condition.")
        continue

    per_stage, global_summary = evaluate_stage_level(
        df_work, models_present, STAGES_CANON, true_col="true_stages"
    )
    per_stage["condition"] = cond_name
    global_summary["condition"] = cond_name

    out_dir.mkdir(parents=True, exist_ok=True)
    per_stage.to_csv(per_path, index=False)
    global_summary.to_csv(glob_path, index=False)

    all_per_stage.append(per_stage)
    all_global.append(global_summary)

if all_per_stage:
    abl2_stage_per_stage = pd.concat(all_per_stage, ignore_index=True)
    abl2_stage_global = pd.concat(all_global, ignore_index=True)
    abl2_stage_per_stage.to_csv(RESULTS_DIR_II / "ALL_stage_level_per_stage.csv", index=False)
    abl2_stage_global.to_csv(RESULTS_DIR_II / "ALL_stage_level_global.csv", index=False)
    print("Saved Ablation II Stage-Level tables.")
else:
    print("No Ablation II results produced.")

Running Ablation II on 200 topic pairs (SAMPLE_N=200)

=== II:minimal_no_evidence ===


II:minimal_no_evidence models (cached): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 93503.59it/s]



=== II:names_only_all ===


II:names_only_all models (cached): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 53970.82it/s]



=== II:full_all ===


...


=== II:full_only_influence ===


II:full_only_influence models (cached): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 184654.89it/s]



=== II:full_only_outcome ===


II:full_only_outcome models (cached): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 160437.86it/s]

Saved Ablation II Stage-Level tables.


In [27]:
RESULTS_DIR_II = outputs_dir / "ablation_ii_context"
path_global_ii = RESULTS_DIR_II / "ALL_stage_level_global.csv"

df_global_ii = pd.read_csv(path_global_ii)
df_global_ii["condition"] = df_global_ii["condition"].astype(str)

base_ii = df_global_ii[df_global_ii["condition"] == "II:full_all"].copy()
f1_full = dict(zip(base_ii["model"], base_ii["F1 Score"]))

df_global_ii["F1Δ_vs_full"] = df_global_ii.apply(
    lambda row: row["F1 Score"] - f1_full.get(row["model"], np.nan),
    axis=1,
)

abl2_with_delta = df_global_ii.copy()
df_nonbase = abl2_with_delta[abl2_with_delta["condition"] != "II:full_all"].copy()

stats_by_model_ii = (
    df_nonbase.groupby("model")["F1Δ_vs_full"]
    .agg(["mean", "std", "min", "max"])
    .sort_index()
    .round(3)
)

table_cond_model_ii = (
    df_nonbase.pivot_table(
        index="condition",
        columns="model",
        values="F1Δ_vs_full",
        aggfunc="mean"
    )
    .sort_index()
    .round(3)
)

print("\nAblation II — ΔF1 vs full_all stats by model (mean/std/min/max)\n")
print(stats_by_model_ii)

print("\nAblation II — ΔF1 vs full_all breakdown by condition × model (mean ΔF1)\n")
print(table_cond_model_ii)


Ablation II — ΔF1 vs full_all stats by model (mean/std/min/max)

                             mean    std    min    max
model                                                 
claude-3-5-haiku-20241022  -0.090  0.174 -0.394  0.134
claude-3-7-sonnet-20250219 -0.377  0.124 -0.619 -0.269
deepseek-chat              -0.406  0.208 -0.666 -0.012
gpt-3.5-turbo              -0.001  0.162 -0.350  0.168
gpt-4o                     -0.300  0.167 -0.594 -0.114
gpt-4o-mini                 0.019  0.157 -0.200  0.333
open-mixtral-8x22b         -0.134  0.196 -0.514  0.064

Ablation II — ΔF1 vs full_all breakdown by condition × model (mean ΔF1)

                         claude-3-5-haiku-20241022  claude-3-7-sonnet-20250219  deepseek-chat  gpt-3.5-turbo  gpt-4o  gpt-4o-mini  open-mixtral-8x22b
condition                                                                                                                                            
II:minimal_no_evidence                      -0.394               

## Key Observations

1. **Models separate cleanly into three robustness tiers under reduced context.**  
   - **Most robust:**  
     - **GPT-3.5-Turbo** (mean −0.001, min −0.350, max +0.168)  
     - **GPT-4o-mini** (mean +0.019, min −0.200, max +0.333)  
     Both show small average shifts and multiple positive ΔF1 values (Turbo +0.168 for `full_only_outcome`; Mini +0.333 for `names_only_all`), indicating resilience to missing contextual structure.  
   - **Moderately robust:**  
     - **Claude Haiku** (mean −0.090, min −0.394)  
     - **Mixtral 8x22B** (mean −0.134, min −0.514)  
     These models degrade consistently but avoid the severe collapses of the weakest tier.  
   - **Least robust:**  
     - **Claude Sonnet** (mean −0.377, min −0.619)  
     - **DeepSeek-Chat** (mean −0.406, min −0.666)  
     - **GPT-4o** (mean −0.300, min −0.594)  
     All show substantial negative shifts across nearly every reduced-context condition.

2. **The `minimal_no_evidence` condition produces the deepest collapse for every model except Turbo.**  
   It yields the most negative ΔF1 values overall:  
   - DeepSeek −0.666  
   - Sonnet −0.619  
   - GPT-4o −0.594  
   - Haiku −0.394  
   - Mixtral −0.514  
   Showing that **topic names and shared concepts alone are insufficient** for meaningful stage inference.

3. **Selective-evidence conditions reveal asymmetric reliance on evidence channels.**  
   - **Outcome-only** and **Influence-only** conditions produce the smallest drops and even positive values for several models  
     (Turbo +0.168; Mixtral +0.064; Haiku +0.019; Mini +0.168).  
   - **Reach-only**, **Engagement-only**, and **Feedback-only** cause sharper declines, particularly for  
     **Sonnet** (−0.419, −0.339, −0.419) and **GPT-4o** (−0.394, −0.194, −0.394).  
   This demonstrates that models depend unequally on different evidence categories.

4. **High-performing models in the full-prompt setting are the least resilient when context is removed.**  
   Sonnet, DeepSeek-Chat, and GPT-4o all show **large negative means** (−0.377, −0.406, −0.300) and the steepest minimum values, indicating strong dependence on full semantic definitions and complete multi-platform evidence.

5. **Lower-capacity models degrade more gradually or even improve under simplified prompts.**  
   Turbo and Mini are the only models with **positive maxima** (+0.168, +0.333) and the smallest average declines, suggesting that while they underperform with full information, they generalise more flexibly when the prompt is simplified.

**Overall:** Ablation II shows that **structured stage definitions and full evidence are critical for the strongest models**, which collapse sharply when this scaffolding is removed. In contrast, **smaller models are less brittle** but also less semantically grounded. Together with Ablation I, these results demonstrate that *robustness to degraded prompts* and *overall accuracy* are distinct dimensions of model behaviour that must be evaluated separately in Flowmetrics applications.

## Overall Summary of Ablation Results

**1. Effect of shuffling stage order (Ablation I).**  
Shuffling the order in which Flowmetrics stages are presented produces clear and model-specific effects. Claude Sonnet and Mixtral remain highly stable, with mean ΔF1 values close to zero and narrow ranges, showing that they rely on semantic distinctions between stages rather than positional cues. GPT-4o and DeepSeek-Chat exhibit moderate sensitivity: their performance drops under adversarial permutations but not uniformly. Haiku, GPT-4o-mini, and GPT-3.5-Turbo are strongly affected, showing large negative shifts or high volatility, which indicates that for these models, ordering information is treated as a structural cue and partially substituted for reasoning over evidence. Overall, the results show that stage-order perturbations weaken models that depend on prompt structure, while semantically robust models remain stable.

**2. Effect of removing context and isolating evidence (Ablation II).**  
Reducing or restricting the prompt context reveals a different sensitivity profile. Models that perform strongly with the full prompt—Sonnet, DeepSeek-Chat, and GPT-4o—exhibit the steepest declines when stage definitions or platform evidence are removed (mean ΔF1 between −0.377 and −0.406). For all models except GPT-3.5-Turbo, the minimal-no-evidence condition causes the largest collapse, confirming that topic names and shared concepts alone are insufficient for meaningful inference. Selective-evidence conditions show that models depend unevenly on different evidence channels: Outcome-only and Influence-only cause the smallest disruptions and sometimes small gains, whereas Reach-only, Engagement-only, and Feedback-only trigger substantial drops for the strongest models. In contrast, GPT-3.5-Turbo and GPT-4o-mini degrade gradually and occasionally improve, indicating a flexible but less principled pattern of generalisation. These results show that high-performing models are also the most dependent on the full semantic scaffold of the prompt.

**3. Relationship between the two ablations.**  
Taken together, the ablations show complementary and consistent patterns. Models that rely strongly on the structured semantics of the Flowmetrics prompt (Sonnet, DeepSeek-Chat, GPT-4o) are robust to stage-order permutations but brittle when context is removed. Conversely, smaller or alignment-tuned models (GPT-4o-mini, GPT-3.5-Turbo) are highly sensitive to stage-order changes but more resilient to reduced context. In other words, robustness to ordering and robustness to context removal are not aligned characteristics. The models that reason most coherently when the full prompt is available are also the ones whose performance collapses most sharply when essential components of that prompt are removed. This divergence underscores that ordering cues and contextual semantics play distinct roles: some models depend on prompt structure for stability, while others depend on semantic richness for accuracy.

**4. Which components contribute most to coherent stage predictions?**  
Across both ablations, three elements emerge as essential for producing stable and interpretable Flowmetrics predictions: (i) **the presence of stage definitions**, without which even top-performing models lose their ability to distinguish stages; (ii) **the availability of multi-stage platform evidence**, since restricting evidence to a single stage leads to asymmetric declines and distorted predictions; and (iii) **the consistency of the stage list structure**, which stabilises weaker models and prevents erratic ordering-sensitive behaviour. These components jointly form the interpretive scaffold that supports coherent stage assignment.

**5. Answers to the guiding questions.**  
(1) *Did shuffling the presented order change sequence metrics in meaningful ways?* Yes. For weaker and volatile models, order changes introduce real signal and shift predictions; for stronger models, changes remain minor.  
(2) *How sensitive are results to minimal context vs. added guidance?* Strong models are extremely sensitive and degrade sharply without full definitions and evidence; weaker ones degrade gradually or sometimes improve.  
(3) *Which pipeline components contribute most to coherent predictions?* Stage definitions, multi-stage evidence, and consistent structural ordering together drive stable and interpretable inference.

**Overall conclusion.**  
The ablations reveal that Flowmetrics inference is shaped by two independent axes of robustness: sensitivity to ordering and sensitivity to contextual richness. Strong models show semantic robustness but contextual brittleness; small models show structural brittleness but contextual flexibility. Effective use of Flowmetrics therefore requires preserving the semantic scaffold of the prompt, maintaining evidence completeness, and avoiding disruptive structural alterations when deploying models in evaluation pipelines.